# Semantic-Agent Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local semantic-agent pipeline for English-to-Chinese cross-lingual dialogue summarization.

The semantic pipeline uses three local small language model agents. Agent 1 reads the original English dialogue and extracts a structured semantic representation, including participants, speaker intent, actor/action/recipient relations, key events, and the final outcome. Agent 2 selects the most salient events and generates a draft Chinese summary. Agent 3 verifies the draft summary against the original dialogue and revises it only if necessary.

```text
English Dialogue
→ Agent 1: Semantic Understanding Agent
→ Agent 2: Event Selection and Chinese Summary Generation Agent
→ Agent 3: Verification and Revision Agent
→ Final Chinese Summary
```

The pipeline consists of three agents:

```text
Agent 1: Semantic Understanding Agent
Input: original English dialogue
Output: structured semantic representation

Agent 2: Event Selection and Chinese Summary Generation Agent
Input: structured semantic representation from Agent 1
Output: selected events and draft Chinese summary

Agent 3: Verification and Revision Agent
Input: original dialogue + Agent 2 selected events and draft summary
Output: revised final Chinese summary
```
This setup is used as the proposed semantic-agent pipeline. Unlike Direct, Translate-then-Summarize, and Summarize-then-Translate baselines, this pipeline explicitly represents pragmatic meaning, semantic roles, actor/action relations, and final dialogue outcomes before generating the final Chinese summary.

The local small language models are served through Ollama. The notebook controls prompt design, JSON parsing, intermediate output inspection, verification/revision, checkpointing, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the model locally.

### Recommended model setup

This notebook uses a single-model multi-agent setup. All three agents use Qwen3.5-27B.

```bash
ollama pull qwen3.5:27b
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [3]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# Semantic-agent models
# All agents use the same 27B model for a consistent model-capacity setting.
SEMANTIC_UNDERSTANDING_MODEL = "qwen3.5:27b"
SUMMARY_GENERATION_MODEL = "qwen3.5:27b"
REVISION_MODEL = "qwen3.5:27b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 4096
REVISION_NUM_CTX = 4096

# Gold set path
GOLD_SET_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_set_50_zh_XSAMSum_bart.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files for semantic-agent pipeline
INTERMEDIATE_OUTPUT_PATH = OUTPUT_DIR / "semantic_qwen27b_all_agents_5samples_intermediate.jsonl"
FULL_OUTPUT_PATH = OUTPUT_DIR / "semantic_qwen27b_all_agents_5samples_final.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "semantic_qwen27b_all_agents_5samples.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "semantic_qwen27b_all_agents_5samples_errors.jsonl"

print("Gold set path:", GOLD_SET_PATH)
print("Output directory:", OUTPUT_DIR)
print("Intermediate JSONL output path:", INTERMEDIATE_OUTPUT_PATH)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

print("Agent 1 model:", SEMANTIC_UNDERSTANDING_MODEL)
print("Agent 2 model:", SUMMARY_GENERATION_MODEL)
print("Agent 3 model:", REVISION_MODEL)

Gold set path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_set_50_zh_XSAMSum_bart.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Intermediate JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples_intermediate.jsonl
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples_final.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples_errors.jsonl
Agent 1 model: qwen3.5:27b
Agent 2 model: qwen3.5:27b
Agent 3 model: qwen3.5:27b


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['qwen3.5:9b', 'qwen3.5:27b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
    keep_alive: Optional[str] = None,
    json_mode: bool = True,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    if json_mode:
        payload["format"] = "json"

    if keep_alive is not None:
        payload["keep_alive"] = keep_alive

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()


def load_ollama_model(model: str) -> None:
    """Preload a model into Ollama memory."""
    payload = {
        "model": model,
        "messages": [],
        "stream": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=300,
    )
    response.raise_for_status()

    print(f"Loaded model: {model}")


def unload_ollama_model(model: str) -> None:
    """Unload a model from Ollama memory."""
    payload = {
        "model": model,
        "messages": [],
        "keep_alive": 0,
        "stream": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=300,
    )
    response.raise_for_status()

    print(f"Unloaded model: {model}")

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [ ]:
# Cell 5: Prompt templates

# Agent 1: Semantic Understanding
SEMANTIC_UNDERSTANDING_PROMPT = """Analyze the English dialogue and extract only the key semantic events needed for the final summary.

Focus on:
- participants
- speaker intent
- actor/action/recipient relations
- final decisions, refusals, commitments, changed plans, or cancellations
- the final outcome of the dialogue

Rules:
- Do not analyze every utterance.
- Ignore greetings, jokes, reactions, and closings unless they affect the outcome.
- Resolve ambiguity from context.
- For imperatives, the actor is usually the listener, not the speaker.
  Example: if A says "Just text him" to B, B is the actor.
- Do not infer visual details from placeholders such as <file_photo>, <file_gif>, or <file>.

Extract 1-6 key events. Use fewer events for simple dialogues.
Write final_outcome as one concise sentence.

Output valid JSON only.

Schema:
{
  "participants": ["speaker names"],
  "semantic_grounding": [
    {
      "event_id": 1,
      "speaker": "speaker name",
      "speech_act": "short label",
      "intended_meaning": "concise interpretation",
      "actor": "person or null",
      "action": "concise action",
      "object": "object or null",
      "recipient": "person or null",
      "evidence": ["short quote"]
    }
  ],
  "final_outcome": "one concise sentence"
}

Input dialogue:
{dialogue}

JSON output:
"""

# Agent 2: Event Selection and Chinese Summary Generation
SUMMARY_GENERATION_PROMPT = """You will receive a structured semantic representation of an English dialogue.

Select the most important events and write a concise Chinese summary.

Rules:
- Select only 1-3 salient events.
- Prioritize the final outcome, changed plans, decisions, refusals, commitments, and important actor/action relations.
- If an earlier plan is later changed, summarize the final updated state.
- Do not invent details from placeholders.
- Use natural Chinese.

Conciseness:
- For simple dialogues, prefer 20-50 Chinese characters.
- For complex dialogues, allow up to 80 Chinese characters.

Output valid JSON only.

Schema:
{
  "selected_events": [
    {
      "event_id": 1,
      "key_event": "short description"
    }
  ],
  "summary_zh": "Chinese summary"
}

Input semantic representation:
{semantic_representation}

JSON output:
"""

# Agent 3: Concise Verification and Revision
REVISION_PROMPT = """Verify the draft Chinese summary against the original English dialogue.

Revise only if necessary.

Check for:
- hallucination
- missing final outcome
- wrong actor/action/recipient
- missing key event
- outdated plan
- unsupported visual detail
- awkward Chinese
- unnecessary verbosity

Rules:
- Trust the original dialogue if it conflicts with selected_events.
- Do not rewrite merely for style.
- If the draft is correct, keep it exactly unchanged.
- Be extremely concise.
- Use short issue tags only.
- revision_reason must be under 12 words.
- Output valid JSON only.

Conciseness:
- For simple dialogues, prefer 20-50 Chinese characters.
- For complex dialogues, allow up to 80 Chinese characters.

Allowed issue tags:
"hallucination", "wrong_actor", "wrong_action", "wrong_recipient",
"missing_final_outcome", "missing_key_event", "outdated_plan",
"too_verbose", "awkward_chinese", "unsupported_visual_detail"

Schema:
{
  "needs_revision": false,
  "issues_identified": [],
  "events_dropped": [],
  "revision_reason": "",
  "summary_zh_final": "Chinese summary"
}

Original English dialogue:
{dialogue}

Draft Chinese summary and selected events:
{summary_points}

JSON output:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: Agent functions

import re

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace only named placeholders while keeping JSON braces in the prompt unchanged."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def extract_json_object(text: str) -> Dict[str, Any]:
    """Extract and parse a JSON object from a model response.

    This function is intentionally robust because small local models may:
    - wrap JSON in markdown fences,
    - add extra text before or after JSON,
    - output Python-style booleans such as True/False.
    """
    raw = text.strip()

    # Remove markdown fences if present.
    if raw.startswith("```"):
        lines = raw.splitlines()
        raw = "\n".join(
            line for line in lines
            if not line.strip().startswith("```")
        ).strip()

    candidates = [raw]

    # Try extracting the substring between the first "{" and the last "}".
    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidates.append(raw[start:end + 1])

    for candidate in candidates:
        # First try direct JSON parsing.
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass

        # Repair common Python-style JSON issues.
        repaired = candidate
        repaired = re.sub(r"\bTrue\b", "true", repaired)
        repaired = re.sub(r"\bFalse\b", "false", repaired)
        repaired = re.sub(r"\bNone\b", "null", repaired)

        try:
            parsed = json.loads(repaired)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass

    # Safe fallback for debugging.
    return {
        "raw_output": text,
        "parse_error": True,
    }


def format_json(data: Any) -> str:
    """Format Python objects as readable JSON text."""
    return json.dumps(
        data,
        ensure_ascii=False,
        indent=2,
    )


def extract_final_chinese_summary(revision_result: Dict[str, Any]) -> str:
    """Extract the final Chinese summary from Agent 3's revision output."""
    summary = revision_result.get("summary_zh_final", "")

    if isinstance(summary, list):
        return " ".join(str(item).strip() for item in summary if str(item).strip())

    if isinstance(summary, str):
        return summary.strip()

    if summary:
        return str(summary).strip()

    # Backward-compatible fallback.
    summary = revision_result.get("summary_zh", "")

    if isinstance(summary, str):
        return summary.strip()

    raw_output = revision_result.get("raw_output", "")

    if isinstance(raw_output, str):
        return raw_output.strip()

    return ""


def semantic_understanding_agent(dialogue: str) -> Dict[str, Any]:
    """Agent 1: Information Extraction Agent.

    Input:
        Original English dialogue

    Output:
        Structured representation of the dialogue
    """
    prompt = fill_prompt(
        SEMANTIC_UNDERSTANDING_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=SEMANTIC_UNDERSTANDING_MODEL,
        prompt=prompt,
        temperature=0.1,
        num_ctx=DEFAULT_NUM_CTX,
        keep_alive="10m",
        json_mode=True,
    )

    return extract_json_object(response)

def summary_generation_agent(
    semantic_representation: Dict[str, Any],
) -> Dict[str, Any]:
    """Agent 2: Summary Generation Agent.

    Input:
        Structured semantic representation from Agent 1

    Output:
        Selected key events and draft Chinese summary
    """
    prompt = fill_prompt(
        SUMMARY_GENERATION_PROMPT,
        {
            "semantic_representation": format_json(semantic_representation),
        },
    )

    response = call_ollama(
        model=SUMMARY_GENERATION_MODEL,
        prompt=prompt,
        temperature=0.2,
        num_ctx=DEFAULT_NUM_CTX,
        keep_alive="10m",
        json_mode=True,
    )

    return extract_json_object(response)


def revision_agent(
    dialogue: str,
    summary_generation_result: Dict[str, Any],
) -> Dict[str, Any]:
    """Agent 3: Verification and Revision Agent.

    Input:
        Original dialogue, selected events, and draft Chinese summary from Agent 2

    Output:
        Revised final Chinese summary
    """
    prompt = fill_prompt(
        REVISION_PROMPT,
        {
            "dialogue": dialogue,
            "summary_points": format_json(summary_generation_result),
        },
    )

    response = call_ollama(
        model=REVISION_MODEL,
        prompt=prompt,
        temperature=0.1,
        num_ctx=REVISION_NUM_CTX,
        keep_alive="10m",
        json_mode=True,
    )

    return extract_json_object(response)

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: Three-agent semantic pipeline

def run_agents_1_2(example: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
    """Run Agent 1 and Agent 2 and save intermediate outputs."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: Semantic Understanding Agent
    semantic_representation = semantic_understanding_agent(dialogue)

    if verbose:
        print("\n" + "=" * 80)
        print(f"Sample ID: {sample_id}")
        print("=== Agent 1 Intermediate Output: Semantic Representation ===")
        print(format_json(semantic_representation))
        print("=" * 80 + "\n")

    # Agent 2: Event Selection and Chinese Summary Generation Agent
    summary_generation = summary_generation_agent(
        semantic_representation=semantic_representation,
    )

    if verbose:
        print("=== Agent 2 Intermediate Output: Selected Events + Draft Chinese Summary ===")
        print(format_json(summary_generation))
        print("=" * 80 + "\n")

    return {
        "id": sample_id,
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,

        # References
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,

        # Models
        "agent1_model": SEMANTIC_UNDERSTANDING_MODEL,
        "agent2_model": SUMMARY_GENERATION_MODEL,

        # Intermediate output from Agent 1
        "agent1_semantic_representation": semantic_representation,
        "agent1_semantic_representation_json": format_json(semantic_representation),

        # Intermediate output from Agent 2
        "agent2_summary_generation": summary_generation,
        "agent2_summary_generation_json": format_json(summary_generation),
    }


def run_agent_3_from_intermediate(
    intermediate_record: Dict[str, Any],
    verbose: bool = True,
) -> Dict[str, Any]:
    """Run Agent 3 using saved Agent 1/2 intermediate outputs."""

    dialogue = intermediate_record["dialogue"]
    summary_generation = intermediate_record["agent2_summary_generation"]

    # Agent 3: Verification and Revision Agent
    revision = revision_agent(
        dialogue=dialogue,
        summary_generation_result=summary_generation,
    )

    final_chinese_summary = extract_final_chinese_summary(
        revision_result=revision,
    )

    if verbose:
        print("=== Agent 3 Output: Verification + Revision ===")
        print(format_json(revision))
        print()
        print("=== Final Chinese Summary ===")
        print(final_chinese_summary)
        print("=" * 80 + "\n")

    return {
        **intermediate_record,

        # Model
        "agent3_model": REVISION_MODEL,

        # Agent 3 output
        "agent3_revision": revision,
        "agent3_revision_json": format_json(revision),

        # Final output
        "agent3_final_chinese_summary": final_chinese_summary,
        "final_summary": final_chinese_summary,

        # Metadata
        "pipeline": "semantic_agent",
        "num_model_calls": 3,
    }


def run_three_agent_pipeline(example: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
    """Run the full semantic-agent pipeline for one example."""

    # Since all agents use the same 27B model, load it once.
    load_ollama_model(SEMANTIC_UNDERSTANDING_MODEL)

    # Agent 1 + Agent 2
    intermediate_record = run_agents_1_2(example, verbose=verbose)

    # Agent 3
    final_record = run_agent_3_from_intermediate(
        intermediate_record,
        verbose=verbose,
    )

    return final_record

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [10]:
# Cell 9: Load first 5 examples from the gold set

def load_examples_from_gold_set(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the first n examples from the gold-set JSON file."""
    if not path.exists():
        raise FileNotFoundError(f"Gold set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"gold_{i+1:05d}",
            "test_index": item.get("test_index", ""),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_gold_set(GOLD_SET_PATH, n=5)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'gold_00001', 'test_index': 59, 'dialogue': "Laura: Where are you?\r\nPaul: Almost there.\r\nLaura: Which is?\r\nPaul: Close to the Mac.\r\nLaura: That's so far away!\r\nPaul: 15 mins\r\nLaura: I am not waiting any more, see you some other time.\r\nPaul: Please, wait!\r\nLaura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.\r\nPaul: I am so sorry.\r\nLaura: I am not. ", 'reference_english_summary': 'Paul is late for a meeting with Laura and she refuses to wait any longer.', 'reference_chinese_summary': '保罗和劳拉见面时迟到了，现在劳拉不想再等了。'}


In [11]:
# Cell 10: Run the semantic-agent pipeline for the first example

result = run_three_agent_pipeline(test_data[0], verbose=True)
result

Loaded model: qwen3.5:27b

Sample ID: gold_00001
=== Agent 1 Intermediate Output: Semantic Representation ===
{
  "participants": [
    "Laura",
    "Paul"
  ],
  "semantic_grounding": [
    {
      "event_id": 1,
      "speaker": "Paul",
      "speech_act": "status update",
      "intended_meaning": "Paul informs Laura of his current location and estimated arrival time.",
      "actor": "Paul",
      "action": "reporting location and ETA",
      "object": "15 minutes away near the Mac",
      "recipient": "Laura",
      "evidence": [
        "Close to the Mac.",
        "15 mins"
      ]
    },
    {
      "event_id": 2,
      "speaker": "Laura",
      "speech_act": "cancellation",
      "intended_meaning": "Laura decides to end the meeting due to excessive waiting time.",
      "actor": "Laura",
      "action": "canceling the meeting",
      "object": "the planned meeting",
      "recipient": "Paul",
      "evidence": [
        "I am not waiting any more, see you some other time."
  

{'id': 'gold_00001',
 'test_index': 59,
 'dialogue': "Laura: Where are you?\r\nPaul: Almost there.\r\nLaura: Which is?\r\nPaul: Close to the Mac.\r\nLaura: That's so far away!\r\nPaul: 15 mins\r\nLaura: I am not waiting any more, see you some other time.\r\nPaul: Please, wait!\r\nLaura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.\r\nPaul: I am so sorry.\r\nLaura: I am not. ",
 'reference_english_summary': 'Paul is late for a meeting with Laura and she refuses to wait any longer.',
 'reference_chinese_summary': '保罗和劳拉见面时迟到了，现在劳拉不想再等了。',
 'agent1_model': 'qwen3.5:27b',
 'agent2_model': 'qwen3.5:27b',
 'agent1_semantic_representation': {'participants': ['Laura', 'Paul'],
  'semantic_grounding': [{'event_id': 1,
    'speaker': 'Paul',
    'speech_act': 'status update',
    'intended_meaning': 'Paul informs Laura of his current location and estimated arrival time.',
    'actor': 'Paul',
    'action': 'reporting location and ETA',
    'object': '1

In [12]:
# Cell 11: Print semantic-agent pipeline result clearly

def print_semantic_agent_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1 Intermediate Output: Semantic Representation ===")
    print(result["agent1_semantic_representation_json"])
    print()

    print("=== Agent 2 Intermediate Output: Selected Events + Draft Chinese Summary ===")
    print(result["agent2_summary_generation_json"])
    print()

    print("=== Agent 3 Output: Verification + Revision ===")
    print(result["agent3_revision_json"])
    print()

    print("=== Final Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Agent 1 model:", result["agent1_model"])
    print("Agent 2 model:", result["agent2_model"])
    print("Agent 3 model:", result["agent3_model"])
    print("Model calls:", result["num_model_calls"])


print_semantic_agent_result(result)

=== Original Dialogue ===
Laura: Where are you?
Paul: Almost there.
Laura: Which is?
Paul: Close to the Mac.
Laura: That's so far away!
Paul: 15 mins
Laura: I am not waiting any more, see you some other time.
Paul: Please, wait!
Laura: I've waited 30 minutes, 15 minutes ago you wrote you were almost here. This is too much.
Paul: I am so sorry.
Laura: I am not. 

=== Agent 1 Intermediate Output: Semantic Representation ===
{
  "participants": [
    "Laura",
    "Paul"
  ],
  "semantic_grounding": [
    {
      "event_id": 1,
      "speaker": "Paul",
      "speech_act": "status update",
      "intended_meaning": "Paul informs Laura of his current location and estimated arrival time.",
      "actor": "Paul",
      "action": "reporting location and ETA",
      "object": "15 minutes away near the Mac",
      "recipient": "Laura",
      "evidence": [
        "Close to the Mac.",
        "15 mins"
      ]
    },
    {
      "event_id": 2,
      "speaker": "Laura",
      "speech_act": "cancell

## 4. Save Results

This saves all intermediate outputs and the final output.


In [13]:
# Cell 12: Reset previous outputs before batch inference

INTERMEDIATE_OUTPUT_PATH.unlink(missing_ok=True)
FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("Intermediate output path:", INTERMEDIATE_OUTPUT_PATH)
print("Final JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
Intermediate output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples_intermediate.jsonl
Final JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples_final.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [ ]:
# Cell 13: Two-stage batch inference with semantic-agent pipeline
# Time stamp: 9m 9.6s

from datetime import datetime

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

subset = test_data[:MAX_EXAMPLES]

print("Batch inference started at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Semantic-agent pipeline with all 27B agents")
print("Agent 1 model:", SEMANTIC_UNDERSTANDING_MODEL)
print("Agent 2 model:", SUMMARY_GENERATION_MODEL)
print("Agent 3 model:", REVISION_MODEL)

# Load the shared 27B model once
load_ollama_model(SEMANTIC_UNDERSTANDING_MODEL)

# -------------------------
# Stage 1: Agent 1 + Agent 2
# -------------------------

processed_intermediate_ids = load_processed_ids(INTERMEDIATE_OUTPUT_PATH)
print(f"Already processed by Agent 1/2: {len(processed_intermediate_ids)} examples")

for ex in tqdm(
    subset,
    desc=f"Stage 1: Agent 1/2 with {SEMANTIC_UNDERSTANDING_MODEL}",
):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_intermediate_ids:
        print(f"Skipping Agent 1/2 already processed sample: {sample_id}")
        continue

    try:
        intermediate_record = run_agents_1_2(ex, verbose=True)

        append_jsonl(intermediate_record, INTERMEDIATE_OUTPUT_PATH)
        processed_intermediate_ids.add(sample_id)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "stage": "agent_1_2",
            "id": sample_id,
            "test_index": ex.get("test_index", ""),
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error in Agent 1/2 on {sample_id}: {repr(e)}")


# -------------------------
# Stage 2: Agent 3
# -------------------------

intermediate_records = load_jsonl(INTERMEDIATE_OUTPUT_PATH)

processed_final_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed by Agent 3: {len(processed_final_ids)} examples")

for intermediate_record in tqdm(
    intermediate_records,
    desc=f"Stage 2: Agent 3 with {REVISION_MODEL}",
):
    sample_id = str(intermediate_record.get("id", "unknown"))

    if sample_id in processed_final_ids:
        print(f"Skipping Agent 3 already processed sample: {sample_id}")
        continue

    try:
        final_record = run_agent_3_from_intermediate(
            intermediate_record,
            verbose=True,
        )

        append_jsonl(final_record, FULL_OUTPUT_PATH)
        processed_final_ids.add(sample_id)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "stage": "agent_3",
            "id": sample_id,
            "test_index": intermediate_record.get("test_index", ""),
            "error": repr(e),
            "dialogue": intermediate_record.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error in Agent 3 on {sample_id}: {repr(e)}")

print("Batch inference finished at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Finished. Intermediate outputs saved to: {INTERMEDIATE_OUTPUT_PATH}")
print(f"Finished. Final outputs saved to: {FULL_OUTPUT_PATH}")

Batch inference started at: 2026-05-08 02:47:57
Semantic-agent pipeline with all 27B agents
Agent 1 model: qwen3.5:27b
Agent 2 model: qwen3.5:27b
Agent 3 model: qwen3.5:27b
Loaded model: qwen3.5:27b
Already processed by Agent 1/2: 0 examples


Stage 1: Agent 1/2 with qwen3.5:27b:   0%|          | 0/5 [00:00<?, ?it/s]


Sample ID: gold_00001
=== Agent 1 Intermediate Output: Semantic Representation ===
{
  "participants": [
    "Laura",
    "Paul"
  ],
  "semantic_grounding": [
    {
      "event_id": 1,
      "speaker": "Paul",
      "speech_act": "status update",
      "intended_meaning": "Paul informs Laura of his current location and estimated arrival time.",
      "actor": "Paul",
      "action": "reporting location and ETA",
      "object": "15 minutes away near the Mac",
      "recipient": "Laura",
      "evidence": [
        "Close to the Mac.",
        "15 mins"
      ]
    },
    {
      "event_id": 2,
      "speaker": "Laura",
      "speech_act": "cancellation",
      "intended_meaning": "Laura decides to end the meeting due to excessive waiting time.",
      "actor": "Laura",
      "action": "canceling the meeting",
      "object": "the planned meeting",
      "recipient": "Paul",
      "evidence": [
        "I am not waiting any more, see you some other time."
      ]
    },
    {
      "

Stage 2: Agent 3 with qwen3.5:27b:   0%|          | 0/5 [00:00<?, ?it/s]

=== Agent 3 Output: Verification + Revision ===
{
  "needs_revision": false,
  "issues_identified": [],
  "events_dropped": [],
  "revision_reason": "",
  "summary_zh_final": "Laura 因已等待 30 分钟且 Paul 仍迟到 15 分钟，决定取消会面并拒绝 Paul 的等待请求，最终离开。"
}

=== Final Chinese Summary ===
Laura 因已等待 30 分钟且 Paul 仍迟到 15 分钟，决定取消会面并拒绝 Paul 的等待请求，最终离开。

=== Agent 3 Output: Verification + Revision ===
{
  "needs_revision": false,
  "issues_identified": [],
  "events_dropped": [],
  "revision_reason": "",
  "summary_zh_final": "Finn 邀请 Zadie 明天去 Elephant and Castle 参观，Zadie 欣然接受。两人最终商定于下午 2 点在购物中心主入口见面。"
}

=== Final Chinese Summary ===
Finn 邀请 Zadie 明天去 Elephant and Castle 参观，Zadie 欣然接受。两人最终商定于下午 2 点在购物中心主入口见面。

=== Agent 3 Output: Verification + Revision ===
{
  "needs_revision": false,
  "issues_identified": [],
  "events_dropped": [],
  "revision_reason": "",
  "summary_zh_final": "Josh 询问购买 iPad 的建议，Brian 推荐三星并提议以更低价格代购。Josh 确认预算后，两人约定下班后通电话以完成交易。"
}

=== Final Chinese Summary ===
Josh 询问购买 iPad 的建议，Brian 推

## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [16]:
# Cell 14: Export semantic-agent summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "test_index": record.get("test_index", ""),
        "dialogue": record.get("dialogue", ""),

        # Final output
        "final_summary": record.get("final_summary", ""),

        # References
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),

        # Intermediate outputs
        "agent1_semantic_representation_json": record.get(
            "agent1_semantic_representation_json", ""
        ),
        "agent2_summary_generation_json": record.get(
            "agent2_summary_generation_json", ""
        ),
        "agent3_revision_json": record.get(
            "agent3_revision_json", ""
        ),

        # Agent 3 final
        "agent3_final_chinese_summary": record.get(
            "agent3_final_chinese_summary", ""
        ),

        # Metadata
        "pipeline": record.get("pipeline", "semantic_agent"),
        "agent1_model": record.get("agent1_model", ""),
        "agent2_model": record.get("agent2_model", ""),
        "agent3_model": record.get("agent3_model", ""),
        "num_model_calls": record.get("num_model_calls", 3),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_qwen27b_all_agents_5samples.csv


,id,test_index,dialogue,final_summary,reference_english_summary,reference_chinese_summary,agent1_semantic_representation_json,agent2_summary_generation_json,agent3_revision_json,agent3_final_chinese_summary,pipeline,agent1_model,agent2_model,agent3_model,num_model_calls
0,gold_00001,59,Laura: Where are you?\r\nPaul: Almost there.\r...,Laura 因已等待 30 分钟且 Paul 仍迟到 15 分钟，决定取消会面并拒绝 Pau...,Paul is late for a meeting with Laura and she ...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。,"{\n ""participants"": [\n ""Laura"",\n ""Pau...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Laura 因已等待 30 分钟且 Paul 仍迟到 15 分钟，决定取消会面并拒绝 Pau...,semantic_agent,qwen3.5:27b,qwen3.5:27b,qwen3.5:27b,3
1,gold_00002,81,Finn: Hey\r\nZadie: Hi there! What's up?\r\nFi...,Finn 邀请 Zadie 明天去 Elephant and Castle 参观，Zadie...,Finn and Zadie are going to Elephant and Castl...,费恩和查蒂明天2点去象堡，他们会在正门碰头。,"{\n ""participants"": [\n ""Finn"",\n ""Zadi...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Finn 邀请 Zadie 明天去 Elephant and Castle 参观，Zadie...,semantic_agent,qwen3.5:27b,qwen3.5:27b,qwen3.5:27b,3
2,gold_00003,85,Josh: I need to buy an iPad?\r\nJosh: do u thi...,Josh 询问购买 iPad 的建议，Brian 推荐三星并提议以更低价格代购。Josh 确...,Josh wants to buy a tablet and doesn't know wh...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...,"{\n ""participants"": [\n ""Josh"",\n ""Bria...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Josh 询问购买 iPad 的建议，Brian 推荐三星并提议以更低价格代购。Josh 确...,semantic_agent,qwen3.5:27b,qwen3.5:27b,qwen3.5:27b,3
3,gold_00004,87,Frank: wat are u doing??\r\nAndy: watching Arr...,Frank 敦促 Andy 立即为测验复习，但 Andy 拒绝并计划明天再学。面对 Fran...,Frank tries to encourage Andy to learn for the...,弗兰克试图激励安迪，为明天的测验学习。,"{\n ""participants"": [\n ""Frank"",\n ""And...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Frank 敦促 Andy 立即为测验复习，但 Andy 拒绝并计划明天再学。面对 Fran...,semantic_agent,qwen3.5:27b,qwen3.5:27b,qwen3.5:27b,3
4,gold_00005,123,Crystal: <file_photo>\r\nIrene: He's so big!\r...,Crystal 表示儿子衣服穿不下了，Irene 主动提出带他去买新衣服，尽管 Crysta...,Irene will take Crystal's son shopping for clo...,艾琳会带克里斯特尔的儿子去买衣服。,"{\n ""participants"": [\n ""Crystal"",\n ""I...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Crystal 表示儿子衣服穿不下了，Irene 主动提出带他去买新衣服，尽管 Crysta...,semantic_agent,qwen3.5:27b,qwen3.5:27b,qwen3.5:27b,3


In [17]:
# Cell 15: Compare semantic-agent outputs with references

comparison_columns = [
    "id",
    "test_index",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,test_index,final_summary,reference_chinese_summary
0,gold_00001,59,Laura 因已等待 30 分钟且 Paul 仍迟到 15 分钟，决定取消会面并拒绝 Pau...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,81,Finn 邀请 Zadie 明天去 Elephant and Castle 参观，Zadie...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,85,Josh 询问购买 iPad 的建议，Brian 推荐三星并提议以更低价格代购。Josh 确...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,87,Frank 敦促 Andy 立即为测验复习，但 Andy 拒绝并计划明天再学。面对 Fran...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,123,Crystal 表示儿子衣服穿不下了，Irene 主动提出带他去买新衣服，尽管 Crysta...,艾琳会带克里斯特尔的儿子去买衣服。


In [18]:
# Cell 16: Inspect semantic-agent intermediate outputs and final output

if not df.empty:
    inspection_columns = [
        "id",
        "test_index",
        "dialogue",
        "agent1_semantic_representation_json",
        "agent2_summary_generation_json",
        "agent3_revision_json",
        "final_summary",
        "reference_english_summary",
        "reference_chinese_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,test_index,dialogue,agent1_semantic_representation_json,agent2_summary_generation_json,agent3_revision_json,final_summary,reference_english_summary,reference_chinese_summary
0,gold_00001,59,Laura: Where are you?\r\nPaul: Almost there.\r...,"{\n ""participants"": [\n ""Laura"",\n ""Pau...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Laura 因已等待 30 分钟且 Paul 仍迟到 15 分钟，决定取消会面并拒绝 Pau...,Paul is late for a meeting with Laura and she ...,保罗和劳拉见面时迟到了，现在劳拉不想再等了。
1,gold_00002,81,Finn: Hey\r\nZadie: Hi there! What's up?\r\nFi...,"{\n ""participants"": [\n ""Finn"",\n ""Zadi...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Finn 邀请 Zadie 明天去 Elephant and Castle 参观，Zadie...,Finn and Zadie are going to Elephant and Castl...,费恩和查蒂明天2点去象堡，他们会在正门碰头。
2,gold_00003,85,Josh: I need to buy an iPad?\r\nJosh: do u thi...,"{\n ""participants"": [\n ""Josh"",\n ""Bria...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Josh 询问购买 iPad 的建议，Brian 推荐三星并提议以更低价格代购。Josh 确...,Josh wants to buy a tablet and doesn't know wh...,乔什想买个平板，但不知道该选哪个牌子。按照布莱恩的说法，其他品牌比苹果更好，他能弄来更便宜的...
3,gold_00004,87,Frank: wat are u doing??\r\nAndy: watching Arr...,"{\n ""participants"": [\n ""Frank"",\n ""And...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Frank 敦促 Andy 立即为测验复习，但 Andy 拒绝并计划明天再学。面对 Fran...,Frank tries to encourage Andy to learn for the...,弗兰克试图激励安迪，为明天的测验学习。
4,gold_00005,123,Crystal: <file_photo>\r\nIrene: He's so big!\r...,"{\n ""participants"": [\n ""Crystal"",\n ""I...","{\n ""selected_events"": [\n {\n ""event...","{\n ""needs_revision"": false,\n ""issues_ident...",Crystal 表示儿子衣服穿不下了，Irene 主动提出带他去买新衣服，尽管 Crysta...,Irene will take Crystal's son shopping for clo...,艾琳会带克里斯特尔的儿子去买衣服。
